# Simple Bayesian logistic regression with PyMC

1. Synthesize patients: age, gender, main complaint, and dead/alive within 30 days.
2. Fit a Bayesian logistic regression with N(0, 1) priors.
3. Check that the posterior recovers the true values the data was generated from.

In [2]:
import os

# must be set before pymc/jax are imported
os.environ["JAX_PLATFORMS"] = "cpu"                                     # GPU was slower for this model
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=6"    # one JAX device per chain, run in parallel

import numpy as np
import pandas as pd
import pymc as pm

SEED = 42


## 1. Synthesize data

- **age**: Normal(55, 18), clipped to 18–100
- **gender**: Bernoulli(0.5), 1 = male
- **complaint**: one of four, equally likely
- **V1, body temperature** (°C): Normal(36.55, 0.4), plus a fever tail for 10% of patients and a hypothermic group of 0.5%
- **V2, systolic blood pressure** (mmHg): log-normal around 131 (sd ≈ 28), rising 0.4 mmHg per year of age above 55
- **V3, heart rate** (bpm): log-normal around 82 (sd ≈ 19), plus 10 bpm per °C of temperature above 36.6

The shapes match the real ED patients in `data.csv.xls` (temperature median 36.5, BP mean 134, heart rate mean 84).
Each vital is then grouped into five levels; `THRESHOLDS` holds the cut-offs and can be edited:

| level | V1 temperature °C | V2 systolic BP mmHg | V3 heart rate bpm |
|---|---|---|---|
| xlow | < 35.0 | < 90 | ≤ 40 |
| low | 35.0–36.0 | 90–99 | 41–50 |
| normal | 36.1–37.9 | 100–139 | 51–90 |
| high | 38.0–38.9 | 140–179 | 91–110 |
| xhigh | ≥ 39.0 | ≥ 180 | ≥ 111 |

- **dead**: Bernoulli(p), where p comes from a logistic model with the true values below.
  The intercept is set so that about 3% die.

Chest pain is the reference complaint, so the other complaints are measured relative to it.

In [3]:
rng = np.random.default_rng(SEED)
N_TRAIN = 8_000
N_TEST = 2_000                   # 80/20 split, about 1,500 test deaths at 3% mortality
N = N_TRAIN + N_TEST
COMPLAINTS = ["chest pain", "shortness of breath", "headache", "abdominal pain"]

df = pd.DataFrame({
    "age": rng.normal(55, 18, N).clip(18, 100).round(),
    "male": rng.binomial(1, 0.5, N),
    "complaint": rng.choice(COMPLAINTS, N),
})
# vital signs, shaped after the real ED patients in data.csv.xls
fever = rng.random(N) < 0.10                      # 10% febrile
cold = rng.random(N) < 0.005                      # 0.5% hypothermic
V1 = rng.normal(36.55, 0.4, N) + fever * rng.gamma(2.0, 0.6, N)
df["V1"] = np.where(cold, rng.normal(34.6, 0.6, N), V1).clip(30, 42.5).round(1)                   # body temperature, °C
df["V2"] = (rng.lognormal(np.log(131), 0.2, N) + 0.4 * (df["age"] - 55)).clip(50, 260).round()     # systolic BP, mmHg
df["V3"] = (rng.lognormal(np.log(82), 0.21, N) + 10 * (df["V1"] - 36.6)).clip(30, 220).round()    # heart rate, bpm

# five levels per vital; each list holds the lower edges of low, normal, high and xhigh
LEVELS = ["xlow", "low", "normal", "high", "xhigh"]
THRESHOLDS = {
    "V1": [35.0, 36.1, 38.0, 39.0],     # temperature, °C
    "V2": [90, 100, 140, 180],          # systolic BP, mmHg
    "V3": [41, 51, 91, 111],            # heart rate, bpm
}
for v, cuts in THRESHOLDS.items():
    df[v + "_cat"] = pd.Categorical(np.array(LEVELS)[np.digitize(df[v], cuts)], categories=LEVELS)
# random train/test split
df["split"] = np.where(rng.permutation(N) < N_TRAIN, "train", "test")
df.head()

,age,male,complaint,V1,V2,V3,V1_cat,V2_cat,V3_cat,split
0,60.0,1,chest pain,36.0,110.0,68.0,low,normal,normal,train
1,36.0,0,chest pain,36.5,131.0,108.0,normal,normal,high,train
2,69.0,1,abdominal pain,36.1,202.0,69.0,normal,xhigh,normal,train
3,72.0,0,headache,38.9,144.0,87.0,high,high,normal,train
4,20.0,0,abdominal pain,36.5,97.0,70.0,normal,low,normal,train


Turn the patients into model inputs. Age is standardised (mean 0, sd 1) so that an
N(0, 1) prior is reasonable for its coefficient. Each complaint except the reference gets a 0/1 column.

In [4]:
train = df["split"] == "train"
test = ~train
# standardise with training statistics only, so nothing about the test set leaks into the fit
AGE_MEAN, AGE_SD = df.loc[train, "age"].mean(), df.loc[train, "age"].std()

X = pd.DataFrame({
    "age": (df["age"] - AGE_MEAN) / AGE_SD,
    "male": df["male"],
})
for c in COMPLAINTS[1:]:
    X[c] = (df["complaint"] == c).astype(int)
# one 0/1 column per vital level; "normal" is the reference
for v in THRESHOLDS:
    for level in LEVELS:
        if level != "normal":
            X[f"{v}_{level}"] = (df[v + "_cat"] == level).astype(int)
X.head()

,age,male,shortness of breath,headache,abdominal pain,V1_xlow,V1_low,V1_high,V1_xhigh,V2_xlow,V2_low,V2_high,V2_xhigh,V3_xlow,V3_low,V3_high,V3_xhigh
0,0.279896,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,-1.085727,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,0.792005,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0
3,0.962707,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0
4,-1.996143,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0


set the true beta and intercept then copmute the log ods of death for each patient and converting that to true probability of death for each patient.

In [6]:
# the true values PyMC should recover
rng = np.random.default_rng(SEED)
TRUE_INTERCEPT = -4.3           # keeps mortality near 3% with the vitals added
TRUE_BETA = pd.Series({
    "age": 0.8,                   # per standard deviation of age (about 18 years)
    "male": 0.3,
    "shortness of breath": 0.5,
    "headache": -0.7,
    "abdominal pain": -0.2,
    # vitals, relative to the normal level of each
    "V1_xlow": 1.0,  "V1_low": 0.3,  "V1_high": 0.4,  "V1_xhigh": 0.8,     # temperature
    "V2_xlow": 1.2,  "V2_low": 0.6,  "V2_high": 0.1,  "V2_xhigh": 0.4,     # systolic BP: low is worse than high
    "V3_xlow": 0.8,  "V3_low": 0.3,  "V3_high": 0.3,  "V3_xhigh": 0.8,     # heart rate
})

p_true = 1 / (1 + np.exp(-(TRUE_INTERCEPT + X @ TRUE_BETA)))
df["dead"] = rng.binomial(1, p_true)
print(f"{df['dead'].sum()} deaths of {N} ({df['dead'].mean():.1%})")
df.groupby("split")["dead"].agg(patients="size", deaths="sum", rate="mean")

321 deaths of 10000 (3.2%)


,patients,deaths,rate
split,,,
test,2000,61,0.0305
train,8000,260,0.0325


### Collapsing identical patients

Every feature is discrete: age in whole years (83 values), sex (2), complaint (4), and three vitals with
five levels each. So there are at most 83 × 2 × 4 × 5³ = 83,000 distinct patient types, however many patients
we simulate, and in practice far fewer occur. Only the training set is collapsed and fitted. Patients of the same type share the same
probability of death $p$, so we fit one row per type with a count, instead of one row per patient.

**The posterior is unchanged.** For a type with $n$ patients of whom $k$ died, the per-patient Bernoulli
likelihood is

$$\prod_{i=1}^{n} p^{y_i}(1-p)^{1-y_i} = p^{k}(1-p)^{n-k},$$

and the Binomial likelihood is $\binom{n}{k}\,p^{k}(1-p)^{n-k}$. They differ only by $\binom{n}{k}$, which does
not depend on the parameters, so both give the same posterior.

**Why do it.** The sampler evaluates the likelihood and its gradient over every row at every step, so the cost
follows the number of rows, not the number of patients. In a benchmark of the model without vitals, 20,000
patients collapsed to 664 rows and the fit went from 27 s to about 6 s, with the same coefficients to 2–3 decimals.
This matters because the updating experiments refit the model many times.

**Why the vitals are grouped.** Continuous vitals make nearly every patient unique, which leaves nothing to
collapse. Grouping them into five clinical levels, as early-warning scores such as NEWS2 do, keeps the number of
types bounded. The model then learns one effect per level (e.g. tachycardia vs normal heart rate) instead of one
slope per unit, and the true values are defined on the same levels so they can still be checked.

**Batches.** The training set is also split into 4 equal batches and collapsed per batch, so section 2.2 can give each updating step new patients. Models that use all batches (2.1) get exactly the same posterior as before.

In [7]:
# each training patient gets one of 4 batches; 2.2 gives each updating step its own batch
df["batch"] = 0
df.loc[train, "batch"] = np.arange(train.sum()) % 4 + 1

# one row per (patient type, batch) with n = patients and dead = deaths
counts = (X[train].assign(dead=df.loc[train, "dead"], batch=df.loc[train, "batch"])
            .groupby(list(X.columns) + ["batch"], as_index=False)
            .agg(n=("dead", "size"), dead=("dead", "sum")))
print(f"{train.sum()} training patients -> {len(counts)} rows")
counts.head()

8000 training patients -> 6268 rows


,age,male,shortness of breath,headache,abdominal pain,V1_xlow,V1_low,V1_high,V1_xhigh,V2_xlow,V2_low,V2_high,V2_xhigh,V3_xlow,V3_low,V3_high,V3_xhigh,batch,n,dead
0,-2.109945,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0
1,-2.109945,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0
2,-2.109945,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,1,0
3,-2.109945,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,2,0
4,-2.109945,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0


## 2. The models

Every model in this section is fitted on the collapsed training rows from section 1. They differ only in what they
are allowed to look at: which features, and which batches of training patients.

    intercept ~ N(0, 1)                  (unless a prior comes from an earlier model, see 2.2)
    beta_j    ~ N(0, 1)                  one per feature the model may use
    p_i       = sigmoid(intercept + x_i · beta)
    dead_i    ~ Bernoulli(p_i)           fitted as dead_t ~ Binomial(n_t, p_t) per collapsed row t

`pm.sample` draws from the posterior with NUTS: NumPyro's JAX-compiled sampler, 6 chains in parallel on the CPU (6 × 667 draws).

### 2.1 All features, all training data

Age, sex, complaint and all three vitals, fitted on the whole training set at once. Sections 3–5 use this model.

In [126]:
with pm.Model(coords={"feature": X.columns}) as model:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta = pm.Normal("beta", mu=0, sigma=1, dims="feature")

    p = pm.math.sigmoid(intercept + pm.math.dot(counts[X.columns].values, beta))
    pm.Binomial("dead", n=counts["n"].values, p=p, observed=counts["dead"].values)

    # 6 chains x 667 draws ≈ 4000 draws, sampled with NumPyro (JAX) on the CPU
    idata = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [intercept, beta]
Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]



















Running chain 0:   0%|          | 0/1667 [00:01<?, ?it/s]









Running chain 0:   5%|▍         | 83/1667 [00:02<00:17, 90.87it/s]









Running chain 0:  10%|▉         | 166/1667 [00:02<00:12, 122.78it/s]









Running chain 0:  15%|█▍        | 249/1667 [00:02<00:08, 160.70it/s]









Running chain 0:  20%|█▉        | 332/1667 [00:03<00:07, 187.17it/s]


Running chain 0:  25%|██▍       | 415/1667 [00:03<00:05, 224.82it/s]









Running chain 0:  30%|██▉       | 498/1667 [00:03<00:04, 237.35it/s]









Running chain 0:  35%|███▍      | 581/1667 [00:04<00:04, 234.61it/s]









Running chain 0:  40%|███▉      | 664/1667 [00:04<00:03, 252.81it/s]









Running chain 0:  45%|████▍     | 747/1667 [00:04<00:03, 246.84it/s]









Running chain 0:  50%|████▉     | 830/1667 [00:05<00:03, 268.60it/s]







Running chain 0:  55%|█████▍    | 913/1667 [00:05<00:02, 

### 2.2 Updating step 1: base model (age, sex, complaint)

Sections 2.2–2.5 build the model one feature group at a time. Each step is fitted on its own batch of training
patients, and its posterior becomes the prior of the next step:

| section | features | prior | data |
|---|---|---|---|
| 2.2 | age, sex, complaint | N(0, 1) | batch 1 |
| 2.3 | + V1 temperature | posterior of 2.2 for the shared parameters, N(0, 1) for the new ones | batch 2 |
| 2.4 | + V2 blood pressure | posterior of 2.3 for the shared parameters, N(0, 1) for the new ones | batch 3 |
| 2.5 | + V3 heart rate | posterior of 2.4 for the shared parameters, N(0, 1) for the new ones | batch 4 |

The prior passed on is a multivariate normal with the mean and covariance of the previous posterior draws, so the
correlations between parameters are carried along.

**Why a new batch each step.** Reusing the same patients would count them twice, once inside the prior and once as
data, and make the posterior overconfident. With one batch per step every training patient is used exactly once, so
the last step (2.5) can be compared directly with 2.1, which saw the same patients all at once.

**What to look for.** When a feature is added, the parameters that were already there change meaning: without the
vitals, the intercept also carries their average risk. So the prior handed to the next step is centred on slightly
wrong values for the larger model, and that step's batch has to correct it.

The table after 2.5 compares the posterior means of every step with the truth and with 2.1.

#### How the update is computed

**Bayes' rule.** For parameters $\theta$ (intercept and betas) and data $D$:

$$p(\theta \mid D) = \frac{p(D \mid \theta)\, p(\theta)}{p(D)} \;\propto\; \underbrace{p(D \mid \theta)}_{\text{likelihood}}\;\underbrace{p(\theta)}_{\text{prior}}$$

**Updating with new data.** If batches $D_1$ and $D_2$ are independent given $\theta$, the likelihood factorises,
$p(D_1, D_2 \mid \theta) = p(D_1 \mid \theta)\,p(D_2 \mid \theta)$, so

$$p(\theta \mid D_1, D_2) \;\propto\; p(D_2 \mid \theta)\;\underbrace{p(D_1 \mid \theta)\,p(\theta)}_{\propto\; p(\theta \mid D_1)}$$

The posterior after $D_1$ is the prior for $D_2$. Done exactly, updating batch by batch gives the same posterior as
fitting all batches at once; this is also why a batch must not be reused, since its likelihood would enter twice.

**What the code does in practice.** The posterior is only available as draws, so it is approximated by a
multivariate normal with the draws' mean $\mu$ and covariance $\Sigma$ (the covariance keeps the correlations
between parameters). When a step adds features, their new parameters $\theta_{\text{new}}$ get an independent
N(0, 1) prior:

$$p(\theta_{\text{old}}, \theta_{\text{new}}) = \mathcal{N}(\theta_{\text{old}};\, \mu, \Sigma)\;\cdot\;\mathcal{N}(\theta_{\text{new}};\, 0, I)$$

**Where it stops being exact.** Two approximations separate step +V3 from 2.1:

1. the normal approximation of each posterior (very accurate here, since logistic-regression posteriors are close to normal with this much data);
2. the model grows between steps. The posterior for $\theta_{\text{old}}$ came from a model without the new features, so it
   describes slightly different quantities (see *What to look for* above), and earlier batches never inform the new parameters.

In [142]:
BASE_COLS = ["age", "male"] + COMPLAINTS[1:]
rows = counts["batch"] == 1

with pm.Model(coords={"feature": BASE_COLS}) as model_base:
    intercept = pm.Normal("intercept", mu=0, sigma=1)
    beta = pm.Normal("beta", mu=0, sigma=1, dims="feature")

    p = pm.math.sigmoid(intercept + pm.math.dot(counts.loc[rows, BASE_COLS].values, beta))
    pm.Binomial("dead", n=counts.loc[rows, "n"].values, p=p, observed=counts.loc[rows, "dead"].values)

    idata_base = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [intercept, beta]
Compiling.. :   0%|          | 0/1667 [00:00<?, ?it/s]



















Running chain 0:   0%|          | 0/1667 [00:01<?, ?it/s]















Running chain 0:  10%|▉         | 166/1667 [00:01<00:01, 1173.43it/s]







Running chain 0:  20%|█▉        | 332/1667 [00:01<00:01, 1322.51it/s]









Running chain 0:  30%|██▉       | 498/1667 [00:01<00:00, 1277.95it/s]












jax.io_callback failed
Traceback (most recent call last):
  File "/home/theo/Work/DEV/skola/pda/.venv/lib/python3.14/site-packages/ipykernel/iostream.py", line 218, in _event_pipe
    event_pipe = self._local.event_pipe
                 ^^^^^^^^^^^^^^^^^^^^^^
AttributeError: '_thread._local' object has no attribute 'event_pipe'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/theo/Work/DEV/skola/pda/.venv/lib/python3.14/site-packages/jax/_src/callback.py", line 437, in io_callback_impl
    return tree_util.

KeyboardInterrupt: 

### 2.3 Updating step 2: + V1 temperature

Prior: the posterior of 2.2 for the parameters it already has, N(0, 1) for the four temperature levels. Data: batch 2 only.

In [129]:
COLS_V1 = BASE_COLS + ["V1_xlow", "V1_low", "V1_high", "V1_xhigh"]
rows = counts["batch"] == 2

# prior for the parameters 2.2 already has: mean and covariance of its posterior draws
prev = np.column_stack([idata_base.posterior["intercept"].values.ravel(),
                        idata_base.posterior["beta"].values.reshape(-1, len(BASE_COLS))])

with pm.Model(coords={"feature": COLS_V1}) as model_v1:
    old = pm.MvNormal("old", mu=prev.mean(axis=0), cov=np.cov(prev, rowvar=False))     # intercept + BASE_COLS
    new = pm.Normal("new", mu=0, sigma=1, shape=len(COLS_V1) - len(BASE_COLS))   # the added levels
    intercept = pm.Deterministic("intercept", old[0])
    beta = pm.Deterministic("beta", pm.math.concatenate([old[1:], new]), dims="feature")

    p = pm.math.sigmoid(intercept + pm.math.dot(counts.loc[rows, COLS_V1].values, beta))
    pm.Binomial("dead", n=counts.loc[rows, "n"].values, p=p, observed=counts.loc[rows, "dead"].values)

    idata_v1 = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [old, new]


+V1: 9 features, batch 2: 2000 patients, 73 deaths


,true,prior mean,posterior mean,posterior sd,2.1 mean,2.1 sd
intercept,-4.3,-3.42,-3.64,0.20,-4.08,0.17
age,0.8,0.91,0.93,0.10,0.85,0.07
male,0.3,-0.17,0.12,0.18,0.25,0.13
shortness of breath,0.5,0.06,0.36,0.22,0.48,0.16
headache,-0.7,-1.43,-0.75,0.31,-0.68,0.21
abdominal pain,-0.2,-0.26,-0.40,0.24,-0.09,0.18
V1_xlow,1.0,0.00,-0.06,0.96,-0.55,0.81
V1_low,0.3,0.00,0.08,0.36,0.15,0.21
V1_high,0.4,0.00,-0.46,0.69,-0.07,0.38
V1_xhigh,0.8,0.00,-0.41,0.84,0.24,0.45


### 2.4 Updating step 3: + V2 blood pressure

Prior: the posterior of 2.3 for the parameters it already has, N(0, 1) for the four blood pressure levels. Data: batch 3 only.

In [130]:
COLS_V2 = COLS_V1 + ["V2_xlow", "V2_low", "V2_high", "V2_xhigh"]
rows = counts["batch"] == 3

# prior for the parameters 2.3 already has: mean and covariance of its posterior draws
prev = np.column_stack([idata_v1.posterior["intercept"].values.ravel(),
                        idata_v1.posterior["beta"].values.reshape(-1, len(COLS_V1))])

with pm.Model(coords={"feature": COLS_V2}) as model_v2:
    old = pm.MvNormal("old", mu=prev.mean(axis=0), cov=np.cov(prev, rowvar=False))     # intercept + COLS_V1
    new = pm.Normal("new", mu=0, sigma=1, shape=len(COLS_V2) - len(COLS_V1))   # the added levels
    intercept = pm.Deterministic("intercept", old[0])
    beta = pm.Deterministic("beta", pm.math.concatenate([old[1:], new]), dims="feature")

    p = pm.math.sigmoid(intercept + pm.math.dot(counts.loc[rows, COLS_V2].values, beta))
    pm.Binomial("dead", n=counts.loc[rows, "n"].values, p=p, observed=counts.loc[rows, "dead"].values)

    idata_v2 = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [old, new]


+V2: 13 features, batch 3: 2000 patients, 59 deaths


,true,prior mean,posterior mean,posterior sd,2.1 mean,2.1 sd
intercept,-4.3,-3.64,-3.80,0.17,-4.08,0.17
age,0.8,0.93,0.89,0.08,0.85,0.07
male,0.3,0.12,0.19,0.15,0.25,0.13
shortness of breath,0.5,0.36,0.51,0.19,0.48,0.16
headache,-0.7,-0.75,-0.67,0.25,-0.68,0.21
abdominal pain,-0.2,-0.40,-0.19,0.21,-0.09,0.18
V1_xlow,1.0,-0.06,-0.37,0.85,-0.55,0.81
V1_low,0.3,0.08,0.11,0.28,0.15,0.21
V1_high,0.4,-0.46,0.19,0.54,-0.07,0.38
V1_xhigh,0.8,-0.41,-0.78,0.71,0.24,0.45


### 2.5 Updating step 4: + V3 heart rate

Prior: the posterior of 2.4 for the parameters it already has, N(0, 1) for the four heart rate levels. Data: batch 4 only.

After this step every training patient has been used exactly once, like in 2.1, so the posterior and 2.1 columns can be compared directly.

In [131]:
COLS_V3 = COLS_V2 + ["V3_xlow", "V3_low", "V3_high", "V3_xhigh"]
rows = counts["batch"] == 4

# prior for the parameters 2.4 already has: mean and covariance of its posterior draws
prev = np.column_stack([idata_v2.posterior["intercept"].values.ravel(),
                        idata_v2.posterior["beta"].values.reshape(-1, len(COLS_V2))])

with pm.Model(coords={"feature": COLS_V3}) as model_v3:
    old = pm.MvNormal("old", mu=prev.mean(axis=0), cov=np.cov(prev, rowvar=False))     # intercept + COLS_V2
    new = pm.Normal("new", mu=0, sigma=1, shape=len(COLS_V3) - len(COLS_V2))   # the added levels
    intercept = pm.Deterministic("intercept", old[0])
    beta = pm.Deterministic("beta", pm.math.concatenate([old[1:], new]), dims="feature")

    p = pm.math.sigmoid(intercept + pm.math.dot(counts.loc[rows, COLS_V3].values, beta))
    pm.Binomial("dead", n=counts.loc[rows, "n"].values, p=p, observed=counts.loc[rows, "dead"].values)

    idata_v3 = pm.sample(667, tune=1000, chains=6, nuts_sampler="numpyro", random_seed=SEED)

NUTS[numpyro]: [old, new]


+V3: 17 features, batch 4: 2000 patients, 63 deaths


,true,prior mean,posterior mean,posterior sd,2.1 mean,2.1 sd
intercept,-4.3,-3.80,-3.84,0.15,-4.08,0.17
age,0.8,0.89,0.86,0.07,0.85,0.07
male,0.3,0.19,0.26,0.13,0.25,0.13
shortness of breath,0.5,0.51,0.49,0.17,0.48,0.16
headache,-0.7,-0.67,-0.65,0.22,-0.68,0.21
abdominal pain,-0.2,-0.19,-0.10,0.18,-0.09,0.18
V1_xlow,1.0,-0.37,-0.47,0.78,-0.55,0.81
V1_low,0.3,0.11,0.04,0.23,0.15,0.21
V1_high,0.4,0.19,0.16,0.44,-0.07,0.38
V1_xhigh,0.8,-0.78,-0.10,0.62,0.24,0.45


#### Comparing the steps

Posterior mean of every parameter after each step, next to the truth and the all-at-once model 2.1 (empty = not in
that model yet). The last two columns compare the uncertainty of 2.1 and of the final step 2.5.

In [ ]:
def draws_of(idata, cols):
    """Posterior draws as a table: one column for the intercept and one per feature."""
    return pd.DataFrame(np.column_stack([idata.posterior["intercept"].values.ravel(),
                                         idata.posterior["beta"].values.reshape(-1, len(cols))]),
                        columns=["intercept"] + cols)

models = {
    "2.1 all at once": draws_of(idata, list(X.columns)),
    "2.2 base": draws_of(idata_base, BASE_COLS),
    "2.3 +V1": draws_of(idata_v1, COLS_V1),
    "2.4 +V2": draws_of(idata_v2, COLS_V2),
    "2.5 +V3": draws_of(idata_v3, COLS_V3),
}

table = pd.DataFrame({"true": pd.Series({"intercept": TRUE_INTERCEPT, **TRUE_BETA})})
for name, d in models.items():
    table[name] = d.mean()
table["sd 2.1"] = models["2.1 all at once"].std()
table["sd 2.5"] = models["2.5 +V3"].std()
table.round(2)

## 3. Posterior vs truth

Each parameter's posterior is 4000 draws. The mean is the best estimate and the
2.5%–97.5% range is the 95% credible interval. The truth should usually fall inside it.

In [132]:
draws = {"intercept": idata.posterior["intercept"].values.ravel()}
beta_draws = idata.posterior["beta"].values.reshape(-1, X.shape[1])
for j, name in enumerate(X.columns):
    draws[name] = beta_draws[:, j]

truth = {"intercept": TRUE_INTERCEPT, **TRUE_BETA}
result = pd.DataFrame({
    "true": truth,
    "mean": {k: v.mean() for k, v in draws.items()},
    "2.5%": {k: np.quantile(v, 0.025) for k, v in draws.items()},
    "97.5%": {k: np.quantile(v, 0.975) for k, v in draws.items()},
})
result["inside"] = (result["2.5%"] <= result["true"]) & (result["true"] <= result["97.5%"])
result.round(2)

,true,mean,2.5%,97.5%,inside
intercept,-4.3,-4.08,-4.42,-3.74,True
age,0.8,0.85,0.71,0.98,True
male,0.3,0.25,-0.00,0.50,True
shortness of breath,0.5,0.48,0.17,0.80,True
headache,-0.7,-0.68,-1.10,-0.26,True
abdominal pain,-0.2,-0.09,-0.44,0.26,True
V1_xlow,1.0,-0.55,-2.23,0.92,False
V1_low,0.3,0.15,-0.26,0.55,True
V1_high,0.4,-0.07,-0.86,0.64,True
V1_xhigh,0.8,0.24,-0.68,1.08,True


## 4. Predict for one patient

A 70-year-old man with shortness of breath. Every posterior draw gives one probability,
so the prediction is a distribution, not a single number.

In [133]:
patient = pd.Series(0.0, index=X.columns)
patient["age"] = (70 - AGE_MEAN) / AGE_SD
patient["male"] = 1
patient["shortness of breath"] = 1

logit_draws = draws["intercept"] + beta_draws @ patient.values
p_draws = 1 / (1 + np.exp(-logit_draws))
p_true_patient = 1 / (1 + np.exp(-(TRUE_INTERCEPT + patient @ TRUE_BETA)))

print(f"true P(death):      {p_true_patient:.3f}")
print(f"posterior mean:     {p_draws.mean():.3f}")
print(f"95% interval:       {np.quantile(p_draws, 0.025):.3f} – {np.quantile(p_draws, 0.975):.3f}")

true P(death):      0.056
posterior mean:     0.068
95% interval:       0.051 – 0.088


## 5. Test set

No model has seen these patients. Each subsection scores one model from section 2. For each test patient the
predicted probability of death is averaged over all posterior draws and compared with what actually happened:

- **AUC**: how well the model ranks patients who died above patients who survived (0.5 = coin flip, 1 = perfect).
- **Brier score**: mean squared error of the predicted probabilities (lower is better).
- **True probabilities**: because the data is simulated we know each patient's real probability of death.
  Scoring those gives the best any model could do, and `mean |p_model − p_true|` shows how close the model gets.
- **Calibration**: within each tenth of predicted risk, predicted and observed death rates should agree.

In [134]:
X_test = X[test]
y_test = df.loc[test, "dead"].values
p_true_test = p_true[test].values
print(f"test patients: {len(y_test)}, deaths: {y_test.sum()}")

def predict_test(d):
    """Posterior mean P(death) per test patient; d = draws with an intercept column and one column per feature."""
    cols = list(d.columns[1:])
    Xt = X_test[cols].values
    a, b = d["intercept"].values, d[cols].values
    p_sum = np.zeros(len(Xt))
    for s in range(0, len(a), 250):                      # in chunks of draws to save memory
        p_sum += (1 / (1 + np.exp(-(a[s:s + 250, None] + b[s:s + 250] @ Xt.T)))).sum(axis=0)
    return p_sum / len(a)

def auc(y, p):
    """Probability that a random death gets a higher prediction than a random survivor."""
    r = pd.Series(p).rank().values
    n1, n0 = y.sum(), len(y) - y.sum()
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def evaluate(d):
    """Print AUC and Brier score for the model with draws d, and return its calibration table."""
    p = predict_test(d)
    print(f"AUC          model {auc(y_test, p):.3f}    true probabilities {auc(y_test, p_true_test):.3f}")
    print(f"Brier score  model {np.mean((p - y_test) ** 2):.4f}   true probabilities {np.mean((p_true_test - y_test) ** 2):.4f}")
    print(f"mean |p_model - p_true|: {np.abs(p - p_true_test).mean():.4f}")

    cal = pd.DataFrame({"predicted": p, "observed": y_test, "true": p_true_test})
    cal["risk tenth"] = pd.qcut(cal["predicted"], 10, labels=False, duplicates="drop") + 1
    return cal.groupby("risk tenth").agg(patients=("observed", "size"), predicted=("predicted", "mean"),
                                         observed=("observed", "mean"), true=("true", "mean")).round(3)

test patients: 2000, deaths: 61


### 5.1 Model 2.1: all features, all training data at once

In [135]:
evaluate(models["2.1 all at once"])

AUC          model 0.689    true probabilities 0.700
Brier score  model 0.0292   true probabilities 0.0292
mean |p_model - p_true|: 0.0061


,patients,predicted,observed,true
risk tenth,,,,
1,200,0.004,0.005,0.004
2,200,0.008,0.005,0.007
3,200,0.011,0.020,0.010
4,200,0.014,0.035,0.013
5,200,0.018,0.005,0.017
6,200,0.024,0.020,0.022
7,200,0.031,0.040,0.028
8,200,0.041,0.055,0.037
9,200,0.061,0.050,0.055


### 5.2 Model 2.2: base model (age, sex, complaint)

In [136]:
evaluate(models["2.2 base"])

AUC          model 0.642    true probabilities 0.700
Brier score  model 0.0295   true probabilities 0.0292
mean |p_model - p_true|: 0.0132


,patients,predicted,observed,true
risk tenth,,,,
1,202,0.003,0.010,0.005
2,205,0.007,0.029,0.009
3,198,0.010,0.020,0.013
4,196,0.014,0.015,0.016
5,201,0.018,0.020,0.019
6,203,0.024,0.030,0.023
7,197,0.031,0.041,0.029
8,200,0.042,0.015,0.041
9,200,0.061,0.040,0.054


### 5.3 Model 2.3: + V1 temperature

In [137]:
evaluate(models["2.3 +V1"])

AUC          model 0.661    true probabilities 0.700
Brier score  model 0.0295   true probabilities 0.0292
mean |p_model - p_true|: 0.0133


,patients,predicted,observed,true
risk tenth,,,,
1,205,0.004,0.005,0.005
2,195,0.008,0.015,0.009
3,202,0.011,0.015,0.012
4,201,0.015,0.040,0.015
5,197,0.019,0.020,0.019
6,202,0.025,0.035,0.023
7,198,0.033,0.025,0.031
8,200,0.045,0.025,0.037
9,203,0.066,0.034,0.054


### 5.4 Model 2.4: + V2 blood pressure

In [141]:
evaluate(models["2.4 +V2"])

AUC          model 0.653    true probabilities 0.700
Brier score  model 0.0292   true probabilities 0.0292
mean |p_model - p_true|: 0.0110


,patients,predicted,observed,true
risk tenth,,,,
1,200,0.004,0.010,0.006
2,200,0.007,0.025,0.009
3,200,0.010,0.015,0.012
4,200,0.013,0.030,0.015
5,201,0.018,0.015,0.018
6,199,0.022,0.005,0.022
7,202,0.030,0.069,0.030
8,198,0.040,0.025,0.040
9,200,0.057,0.015,0.052


### 5.5 Model 2.5: + V3 heart rate (all features, updated step by step)

In [139]:
evaluate(models["2.5 +V3"])

AUC          model 0.690    true probabilities 0.700
Brier score  model 0.0290   true probabilities 0.0292
mean |p_model - p_true|: 0.0109


,patients,predicted,observed,true
risk tenth,,,,
1,200,0.004,0.015,0.005
2,201,0.008,0.005,0.008
3,199,0.012,0.015,0.012
4,202,0.015,0.025,0.014
5,198,0.020,0.025,0.017
6,200,0.025,0.015,0.023
7,200,0.033,0.030,0.029
8,200,0.043,0.040,0.037
9,200,0.061,0.035,0.057
